# Two-Pass PDE Model Derivation

Derives shallow water moment equations from the **full 3D Incompressible Navier-Stokes**
for **any vertical basis**: Legendre, Chebyshev, B-splines, Galerkin (Shen-type).

| Pass | What | Output |
|------|------|--------|
| **Pass 1** | Basis-independent: INS + material + hydrostatic + BCs + term tagging | `PreProjectedEquations` |
| **Pass 2** | Basis-specific: ansatz substitution, Galerkin projection, M^{-1} | `ProjectedModel` (a `Model`) |

This notebook walks through both passes and compares results across bases.

In [ ]:
import sympy as sp
from sympy import Symbol, Derivative, Rational, S, sqrt, Matrix, latex
sp.init_printing()

---
## Part 1: The INS Generator (building blocks)

Everything starts from the **full 3D INS**. The `StateSpace` holds shared symbols.
`Expression` objects support `.terms`, `.apply()`, `.ibp()`, `.project()`.
See the companion notebook `pde_generator_design.py` for details.

In [ ]:
from zoomy_core.model.models.ins_generator import (
    StateSpace, FullINS, Expression, IBPResult,
    materials, assumptions, Newtonian, Inviscid,
)

state = StateSpace(dimension=1)
ins = FullINS(state)

print("Full 3D INS equations (1D horizontal):")
for eq in ins.equations:
    print(f"  {eq.name}: {len(eq)} terms")

### Material models and assumptions

Materials and assumptions are `Relation` objects (substitution rules).
They take the `StateSpace`, NOT the equations -- they're independent.

In [ ]:
newton = materials.newtonian(state)
inv = materials.inviscid(state)
hydro = assumptions.hydrostatic_pressure(state)

print("Newtonian stress (6 rules):")
display(newton)

print("\nHydrostatic pressure:")
display(hydro)

---
## Part 2: Pass 1 -- Basis-Independent Derivation

`derive_shallow_moments()` does the full derivation from INS:

1. Apply material model (Newtonian by default)
2. Apply hydrostatic assumption: simplify z-momentum -> p(z)
3. Substitute hydrostatic pressure into x-momentum
4. Depth-integrate with IBP on z-derivatives
5. Apply kinematic BCs to boundary terms
6. Apply stress BCs (Navier-slip at bottom, free at surface)
7. Tag each term: `temporal`, `flux`, `nonconservative`, `source`

In [ ]:
from zoomy_core.model.models.model_derivation import (
    derive_shallow_moments, PreProjectedEquations, TaggedTerm,
)

pre = derive_shallow_moments(state, material=Newtonian(state))

print(pre.summary())

### Inspecting tagged terms

Each term carries a `role` (how it maps to the `Model` interface)
and an `origin` (which physical mechanism produced it).

In [ ]:
print("=== Continuity ===")
for t in pre.continuity:
    print(f"  {t.role:20s}  origin={t.origin:20s}  expr={t.expr.expr}")

print("\n=== X-Momentum ===")
for t in pre.x_momentum:
    print(f"  {t.role:20s}  origin={t.origin:20s}  expr={str(t.expr.expr)[:60]}")

The key insight: **these terms are still abstract** -- they contain `u(t,x,z)`, `w(t,x,z)`,
not yet expanded into basis coefficients. This is what makes Pass 1 basis-independent.

The tagged roles map directly to `Model` methods:

| Role | Model method | Physical meaning |
|------|-------------|-----------------|
| `temporal` | mass matrix M | d(h alpha)/dt |
| `flux` | `flux()` | d(F)/dx |
| `nonconservative` | `nonconservative_matrix()` | B(Q) . dQ/dx |
| `source` | `source()` | algebraic in Q |

### Inviscid variant

Switch the material model to see how the tagged terms change.

In [ ]:
pre_inv = derive_shallow_moments(state, material=Inviscid(state))
print(pre_inv.summary())

print("\nCompare x-momentum terms:")
print(f"  Newtonian: {len(pre.x_momentum)} terms")
print(f"  Inviscid:  {len(pre_inv.x_momentum)} terms")

newton_origins = {t.origin for t in pre.x_momentum}
inv_origins = {t.origin for t in pre_inv.x_momentum}
print(f"  Missing in inviscid: {newton_origins - inv_origins}")

---
## Part 3: Pass 2 -- Basis-Specific Projection

`ProjectedModel` takes `PreProjectedEquations` + a basis + level and produces
a complete `Model` with `flux()`, `source()`, `eigenvalues()`, etc.

The projection:
1. Substitutes ansatz: $u(\zeta) = \sum_k \alpha_k \varphi_k(\zeta)$
2. Projects each tagged term onto test functions $\varphi_j$
3. Collects raw projected vectors (with mass matrix M in front)
4. Applies $M^{-1}$ once to ALL non-temporal terms
5. Outputs `Model`-compatible `flux()`, `source()`, `nonconservative_matrix()`

In [ ]:
from zoomy_core.model.models.projected_model import ProjectedModel
from zoomy_core.model.models.basisfunctions import (
    Legendre_shifted, Chebyshevu_shifted, SplineBasis, GalerkinBasis,
)

### 3a. Legendre basis (standard)

Legendre on [0,1]: $\varphi_k(\zeta) = P_k(2\zeta - 1) \cdot (-1)^k$

- Orthogonal: $M$ is diagonal
- $\varphi_0 = 1$ (constant), so mean coefficients = $[1, 0, 0, \ldots]$
- $M^{-1}$ is trivially $\mathrm{diag}(1/M_{kk})$

In [ ]:
leg = ProjectedModel(pre, basis_type=Legendre_shifted, level=2)

print(f"Variables: {list(leg.variables.keys())}")
print(f"n_variables: {leg.n_variables}")
print(f"Mean coefficients: {leg.mean_coefficients()}")

print("\nMass matrix M:")
display(leg.mass_matrix())

print("\nM^{-1}:")
display(leg.mass_matrix_inverse())

#### Legendre flux

In [ ]:
F_leg = leg.flux()
print("Legendre L2 flux (x-direction):")
for i in range(leg.n_variables):
    print(f"  F[{i}] = {F_leg[i, 0]}")

#### Legendre hydrostatic pressure (separated for well-balanced schemes)

In [ ]:
P_leg = leg.hydrostatic_pressure()
print("Hydrostatic pressure:")
for i in range(leg.n_variables):
    val = P_leg[i, 0]
    if val != 0:
        print(f"  P[{i}] = {val}")

#### Legendre non-conservative matrix (topography coupling)

In [ ]:
NC_leg = leg.nonconservative_matrix()
print("Non-conservative Bx (non-zero entries):")
for r in range(leg.n_variables):
    for c in range(leg.n_variables):
        val = NC_leg[r, c, 0]
        if val != 0:
            print(f"  Bx[{r},{c}] = {val}")

#### Legendre source terms (viscosity + slip friction)

In [ ]:
visc_leg = leg.newtonian()
slip_leg = leg.slip()

print("Newtonian viscous source:")
for i in range(leg.n_variables):
    val = visc_leg[i]
    if val != 0:
        print(f"  S_visc[{i}] = {val}")

print("\nNavier-slip friction:")
for i in range(leg.n_variables):
    val = slip_leg[i]
    if val != 0:
        print(f"  S_slip[{i}] = {val}")

### 3b. B-Spline basis

Raw B-splines on [0,1] with hat functions:
- Partition of unity: $\sum_k B_k = 1$ -> mean coefficients = $[1, 1, 1, \ldots]$
- **Non-diagonal** mass matrix M
- $M^{-1}$ is a full matrix -- redistributes contributions across modes

In [ ]:
spl = ProjectedModel(pre, basis_type=SplineBasis, level=2)

print(f"Mean coefficients: {spl.mean_coefficients()}")
print("\nMass matrix M:")
display(spl.mass_matrix())
print("\nM^{-1}:")
display(spl.mass_matrix_inverse())

#### Spline flux

Compare with Legendre: the mass flux is $h \cdot (\alpha_0 + \alpha_1 + \alpha_2)$
because all spline coefficients contribute equally to the mean velocity
(partition of unity).

In [ ]:
F_spl = spl.flux()
print("SplineBasis L2 flux (x-direction):")
for i in range(spl.n_variables):
    expr = sp.simplify(F_spl[i, 0])
    print(f"  F[{i}] = {expr}")

#### Spline source: M^{-1} redistribution

For splines, slip friction at the bottom ($\zeta = 0$) only directly excites
$\varphi_0$ (which peaks at the bottom). But $M^{-1}$ redistributes this
across all modes.

In [ ]:
slip_spl = spl.slip()
print("Spline Navier-slip (with M^{-1} redistribution):")
for i in range(spl.n_variables):
    val = slip_spl[i]
    if val != 0:
        print(f"  S_slip[{i}] = {sp.simplify(val)}")

### 3c. Chebyshev U basis (shifted to [0,1])

$\varphi_k(\zeta) = U_k(2\zeta - 1)$ with weight $w(\zeta) = \sqrt{\zeta(1-\zeta)}$.

- Orthogonal: diagonal $M$ (like Legendre)
- But $M_{kk} = \pi/8$ (not simple rationals)
- $\varphi_0 = 1$, same SWE limit as Legendre
- **Caveat**: weight vanishes at boundaries -> boundary terms are killed

In [ ]:
cheb = ProjectedModel(pre, basis_type=Chebyshevu_shifted, level=2)

print(f"Mean coefficients: {cheb.mean_coefficients()}")
print("\nMass matrix M:")
display(cheb.mass_matrix())
print("\nM^{-1}:")
display(cheb.mass_matrix_inverse())

#### Chebyshev boundary values

The weight $\sqrt{\zeta(1-\zeta)}$ vanishes at $\zeta=0$ and $\zeta=1$.
This means slip friction and kinematic BC boundary terms are killed
in the weighted inner product. This is a known limitation of weighted bases.

In [ ]:
cheb_basis = Chebyshevu_shifted(level=2)
from zoomy_core.model.models.symbolic_integrator import SymbolicIntegrator

si = SymbolicIntegrator(cheb_basis)
mats = si.compute_all_matrices(2)
print("Chebyshev boundary values phi_k(0):")
for k in range(3):
    print(f"  phi_{k}(0) = {mats['phib'][k]}")

### 3d. Galerkin (Shen-type) basis

BC-aware basis via recombination of parent polynomials:
- $\varphi_0 = 1$ (constant, always)
- $\varphi_k$ (k >= 1): built to satisfy boundary conditions

Supported BCs: `noslip`, `nostress`, `slip`, `free`

In [ ]:
gal_basis = GalerkinBasis(level=2, parent="legendre",
                           bc_bottom="slip", bc_top="nostress",
                           slip_length=0.5)

print(f"Galerkin basis ({gal_basis.name}):")
for k in range(3):
    print(f"  phi_{k} = {gal_basis.get(k)}")

In [ ]:
gal = ProjectedModel(pre, basis_type=GalerkinBasis, level=2)

print("Galerkin M:")
display(gal.mass_matrix())

---
## Part 4: Comparing Bases

### Mass matrix structure

In [ ]:
bases = {
    "Legendre": (Legendre_shifted, 2),
    "SplineBasis": (SplineBasis, 2),
    "Chebyshev U": (Chebyshevu_shifted, 2),
}

print("Mass matrix comparison (level 2):")
print("=" * 60)
for name, (cls, lvl) in bases.items():
    m = ProjectedModel(pre, basis_type=cls, level=lvl)
    M = m.mass_matrix()
    is_diag = M == sp.diag(*[M[i, i] for i in range(3)])
    print(f"\n{name} (diagonal={is_diag}):")
    display(M)

### Mean coefficients

The depth-averaged velocity is $\bar{u} = \sum_k c_k \alpha_k$.

In [ ]:
print("Mean coefficients (level 2):")
for name, (cls, lvl) in bases.items():
    m = ProjectedModel(pre, basis_type=cls, level=lvl)
    c = m.mean_coefficients()
    print(f"  {name:15s}: c = {c}")

### Flux structure at level 0 (SWE limit)

At level 0, bases with $\varphi_0 = 1$ (Legendre, Chebyshev) recover
the standard shallow water equations exactly:
- Mass: $F_1 = h u$
- Momentum: $F_2 = h u^2$
- Pressure: $P_2 = g e_z h^2 / 2$

**Note**: SplineBasis L0 uses $B_0 = 1 - \zeta$ (linear hat), not a constant.
It needs L1+ (two hats) to represent uniform flow. The factors differ by
$A_{000}/M_{00}$ which for a linear hat gives $3/4$ instead of $1$.

In [ ]:
print("Level 0 flux comparison (SWE limit):")
print("=" * 60)
for name, cls in [("Legendre", Legendre_shifted), ("SplineBasis", SplineBasis)]:
    m0 = ProjectedModel(pre, basis_type=cls, level=0)
    F = m0.flux()
    P = m0.hydrostatic_pressure()
    print(f"\n{name} (phi_0 = {cls(level=0).get(0)}):")
    print(f"  Mass flux:     F[1] = {F[1, 0]}")
    print(f"  Mom. flux:     F[2] = {F[2, 0]}")
    print(f"  Pressure:      P[2] = {P[2, 0]}")

---
## Part 5: Building a Complete Model for Simulation

To run a simulation, we need:
1. A `ProjectedModel` with source terms (gravity + viscosity + friction)
2. A `NumericalModel` wrapper for regularization
3. A solver (explicit or IMEX)

Here's how to build an inclined plane model:

In [ ]:
from zoomy_core.misc.misc import ZArray

class InclinedPlaneProjected(ProjectedModel):
    """Inclined plane: gravity (g*ez) + Newtonian viscosity + Navier-slip."""

    def source(self):
        p = self.parameters
        h = self.variables[1]
        S = ZArray.zeros(self.n_variables)
        n_mom = self.level + 1
        # Gravity source: g*ez*h projected via Galerkin integral
        # raw[l] = g*ez*h * integral(phi_l) = g*ez*h * (M @ c_mean)[l]
        phi_int = self._phi_int
        raw_grav = [p.g * p.ez * h * phi_int[l] for l in range(n_mom)]
        for k in range(n_mom):
            S[2 + k] = self._apply_Minv(raw_grav, k)
        # Viscosity + slip friction
        visc = self.newtonian()
        slip = self.slip()
        for i in range(self.n_variables):
            S[i] = S[i] + visc[i] + slip[i]
        return S

### Inspect the inclined plane source

In [ ]:
ip_leg = InclinedPlaneProjected(pre, basis_type=Legendre_shifted, level=2,
                                 eigenvalue_mode="numerical")
S_leg = ip_leg.source()

print("Inclined plane Legendre L2 source:")
for i in range(ip_leg.n_variables):
    val = S_leg[i]
    if val != 0:
        print(f"  S[{i}] = {sp.simplify(val)}")

The gravity term $g \cdot e_z \cdot h$ enters only the mean-velocity mode (index 2)
for Legendre because $c = [1, 0, 0]$. Viscosity acts on modes 1+ (proportional to
$\nu / h^2$), and slip friction couples all modes through $\varphi_k(0)$ boundary values.

### SplineBasis inclined plane

For splines, gravity is distributed across ALL modes because $c = [1, 1, 1]$
and then $M^{-1}$ redistributes further.

In [ ]:
ip_spl = InclinedPlaneProjected(pre, basis_type=SplineBasis, level=2,
                                 eigenvalue_mode="numerical")
S_spl = ip_spl.source()

print("Inclined plane SplineBasis L2 source:")
for i in range(ip_spl.n_variables):
    val = S_spl[i]
    if val != 0:
        print(f"  S[{i}] = {sp.simplify(val)}")

---
## Part 6: Architecture Summary

```
Pass 1 (basis-independent)
--------------------------
FullINS(state)
  |
  +-- .apply(material)     -> Newtonian / Inviscid stress
  +-- .apply(hydrostatic)  -> p = rho*g*(eta - z)
  |
  v
derive_shallow_moments()
  |
  +-- depth integration with IBP on d/dz terms
  +-- kinematic BCs on boundary terms
  +-- stress BCs (slip at bottom, free at top)
  +-- tag each term: temporal / flux / NC / source
  |
  v
PreProjectedEquations  (cached, reusable for ANY basis)


Pass 2 (basis-specific)
-----------------------
PreProjectedEquations + Basis + Level
  |
  +-- substitute ansatz: u(z) = sum alpha_k phi_k(z)
  +-- project onto phi_j via SymbolicIntegrator
  +-- compute M, A, D, phib matrices
  +-- apply M^{-1} to all non-temporal terms
  |
  v
ProjectedModel(Model)
  |
  +-- flux()                    -> advection + mass flux
  +-- hydrostatic_pressure()    -> g*h^2/2 (separated for well-balanced)
  +-- nonconservative_matrix()  -> topography + vertical coupling
  +-- source() (overridable)    -> gravity + viscosity + friction
  +-- eigenvalues()             -> symbolic or numerical
  |
  v
NumericalModel(ProjectedModel)  -> regularized for simulation
  |
  v
Solver (explicit / IMEX)
```

The **same** `PreProjectedEquations` feeds into different bases.
The basis only enters in Pass 2.